# 17a — Kohonen's Self-Organising Map

Corresponds to: Note 17 — Kohonen's Self-Organising Map

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. BMU selection — Euclidean distance, verified against Note 17 Part 3 worked example
2. Neighbourhood function and weight update — verified against Note 17 Part 4 worked example
3. Learning rate and neighbourhood decay — exponential schedules
4. Full SOM training on 2D data from 3 clusters — weight vectors converge to cover the data distribution

In [ ]:
import numpy as np

---
## Part 1: BMU Selection

The BMU is the neuron whose weight vector is closest to the input:

    j* = argmin_j ||x − wⱼ||

We reproduce the worked example from Note 17 Part 3:
x = (0.7, 0.4), w₁=(0.2,0.8), w₂=(0.6,0.3), w₃=(0.9,0.1).

In [ ]:
x = np.array([0.7, 0.4])
W = np.array([[0.2, 0.8],
              [0.6, 0.3],
              [0.9, 0.1]])

dists = np.linalg.norm(W - x, axis=1)
bmu   = np.argmin(dists)

print('BMU selection (Note 17 Part 3):')
for j, (w, d) in enumerate(zip(W, dists), 1):
    marker = '  ← BMU' if j - 1 == bmu else ''
    print(f'  j={j}: w={w}, ||x-w|| = {d:.4f}{marker}')

print()
print('Expected: j=2 is BMU (distance 0.1414)')

---
## Part 2: Neighbourhood Function and Weight Update

The Gaussian neighbourhood function:

    h(j, j*) = exp(−d(j,j*)² / 2σ²)

Weight update:

    Δwⱼ = η · h(j, j*) · (x − wⱼ)

We reproduce the Note 17 Part 4 worked example: σ=1.5, η=0.5, BMU=j=2.

In [ ]:
sigma = 1.5
eta   = 0.5

# Grid positions for a 1D map: j=0,1,2
grid_pos = np.array([0, 1, 2])
bmu_pos  = grid_pos[bmu]

# Grid distances and neighbourhood strengths
grid_dists = np.abs(grid_pos - bmu_pos)
h = np.exp(-grid_dists**2 / (2 * sigma**2))

print('Neighbourhood strengths (σ=1.5):')
for j, (d, hj) in enumerate(zip(grid_dists, h), 1):
    print(f'  h({j},{bmu+1}): grid_dist={d}, h = {hj:.4f}')

print()
print('Expected: h(1,2)=0.8007, h(2,2)=1.0000, h(3,2)=0.8007')

# Weight updates
W_new = W.copy().astype(float)
print('\nWeight updates:')
for j in range(len(W)):
    dw = eta * h[j] * (x - W[j])
    W_new[j] = W[j] + dw
    print(f'  dw{j+1} = {dw.round(4)},  w{j+1}_new = {W_new[j].round(4)}')

print()
print('Expected:')
print('  w1_new = (0.4002, 0.6399)')
print('  w2_new = (0.6500, 0.3500)')
print('  w3_new = (0.8199, 0.2201)')

---
## Part 3: Learning Rate and Neighbourhood Decay

Both η and σ decay exponentially over training:

    η(t) = η₀ · exp(−t / τ_η)
    σ(t) = σ₀ · exp(−t / τ_σ)

We show how both parameters evolve over 1000 training steps.

In [ ]:
T      = 1000
eta0   = 0.5
sigma0 = 3.0
eta_f  = 0.01
sigma_f = 0.5

# Time constants so that parameter reaches final value at t=T
tau_eta   = T / np.log(eta0   / eta_f)
tau_sigma = T / np.log(sigma0 / sigma_f)

print(f'Decay schedule over {T} steps:')
print(f'  η:  {eta0} → {eta_f}   (τ = {tau_eta:.1f})')
print(f'  σ:  {sigma0} → {sigma_f}  (τ = {tau_sigma:.1f})')
print()
print(f'{"Step":>6}  {"η(t)":>8}  {"σ(t)":>8}  {"Phase"}')
print('-' * 42)
for t in [0, 100, 200, 400, 600, 800, 999]:
    eta_t   = eta0   * np.exp(-t / tau_eta)
    sigma_t = sigma0 * np.exp(-t / tau_sigma)
    phase = 'ordering' if sigma_t > 1.0 else 'convergence'
    print(f'{t:>6}  {eta_t:>8.4f}  {sigma_t:>8.4f}  {phase}')

---
## Part 4: Full SOM Training on 2D Data

We train a 5×5 SOM on 2D data drawn from 3 Gaussian clusters.
After training, the weight vectors should cover the data distribution,
with nearby neurons responding to similar inputs.

We track quantisation error (average distance from each input to its BMU)
to confirm convergence.

In [ ]:
rng = np.random.default_rng(42)

# 3 clusters of 2D data
c1 = rng.normal([1.0, 1.0], 0.3, (100, 2))
c2 = rng.normal([4.0, 1.0], 0.3, (100, 2))
c3 = rng.normal([2.5, 4.0], 0.3, (100, 2))
data = np.vstack([c1, c2, c3])

print(f'Dataset: {len(data)} samples, 2D, 3 clusters')
print(f'  Cluster 1 centre: ~(1.0, 1.0)')
print(f'  Cluster 2 centre: ~(4.0, 1.0)')
print(f'  Cluster 3 centre: ~(2.5, 4.0)')

In [ ]:
# SOM setup: 5x5 grid, 2D input
map_rows, map_cols = 5, 5
n_neurons = map_rows * map_cols
n_input   = 2

# Grid positions for each neuron
grid_r = np.array([i // map_cols for i in range(n_neurons)])
grid_c = np.array([i %  map_cols for i in range(n_neurons)])

# Initialise weights uniformly within data range
W_som = rng.uniform([data.min(axis=0)], [data.max(axis=0)], (n_neurons, n_input))

# Training parameters
T_train = 2000
eta0, eta_f     = 0.5, 0.01
sigma0, sigma_f = 3.0, 0.5
tau_eta   = T_train / np.log(eta0   / eta_f)
tau_sigma = T_train / np.log(sigma0 / sigma_f)

def quantisation_error(W, data):
    errors = []
    for x in data:
        dists = np.linalg.norm(W - x, axis=1)
        errors.append(dists.min())
    return np.mean(errors)

print(f'Initial quantisation error: {quantisation_error(W_som, data):.4f}')
print()
print(f'{"Step":>6}  {"Quant. error":>14}  {"η":>8}  {"σ":>8}')
print('-' * 44)

for t in range(T_train):
    eta_t   = eta0   * np.exp(-t / tau_eta)
    sigma_t = sigma0 * np.exp(-t / tau_sigma)

    # Pick random input
    x = data[rng.integers(len(data))]

    # Find BMU
    dists = np.linalg.norm(W_som - x, axis=1)
    bmu_idx = np.argmin(dists)

    # Grid distances from BMU
    dr = grid_r - grid_r[bmu_idx]
    dc = grid_c - grid_c[bmu_idx]
    grid_d2 = dr**2 + dc**2

    # Neighbourhood strengths
    h = np.exp(-grid_d2 / (2 * sigma_t**2))

    # Update all neurons
    W_som += eta_t * h[:, np.newaxis] * (x - W_som)

    if t in [0, 99, 499, 999, 1999]:
        qe = quantisation_error(W_som, data)
        print(f'{t+1:>6}  {qe:>14.4f}  {eta_t:>8.4f}  {sigma_t:>8.4f}')

print(f'\nFinal quantisation error: {quantisation_error(W_som, data):.4f}')

Show the final weight vectors — they should cluster near the three data cluster centres.

In [ ]:
print('Final weight vectors (5x5 grid, showing w[:,0] and w[:,1]):')
print(f'{"Neuron":>8}  {"Grid pos":>10}  {"w[0]":>8}  {"w[1]":>8}')
print('-' * 42)
for i in range(n_neurons):
    print(f'{i+1:>8}  ({grid_r[i]},{grid_c[i]}){" ":>6}  {W_som[i,0]:>8.3f}  {W_som[i,1]:>8.3f}')

print()
print('Data cluster centres for reference:')
print('  Cluster 1: ~(1.0, 1.0)')
print('  Cluster 2: ~(4.0, 1.0)')
print('  Cluster 3: ~(2.5, 4.0)')
print()
print('Weight vectors should be distributed across the data range,')
print('with nearby neurons holding similar weight vectors.')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| BMU selection | j* = argmin ||x − wⱼ|| — verified against Note 17 Part 3: BMU = j=2 |
| Neighbourhood function | h(j,j*) = exp(−d²/2σ²) — verified against Note 17 Part 4: h=0.8007 at d=1, σ=1.5 |
| Weight update | Δwⱼ = η·h·(x−wⱼ) — all three neurons move toward x, BMU moves most |
| Decay schedule | η and σ decay exponentially — ordering phase (large σ) transitions to convergence phase (small σ) |
| Full training | Quantisation error decreases over 2000 steps — weight vectors converge to cover the 3-cluster data distribution |